In [0]:
%run ../common/config

In [0]:
cliams_df= spark.read.table(f"{env_catalog}.bronze.claims")

In [0]:
display(cliams_df)

In [0]:
cliams_df.printSchema()

In [0]:
display(claims_df)

In [0]:
claims_trans_df=spark.read.table(f"{env_catalog}.bronze.claims_transactions")

In [0]:
display(claims_trans_df)

In [0]:
claims_trans_df.printSchema()

In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    current_timestamp,
    lit
)

In [0]:
silver_claims = (
    claims_df
    .dropDuplicates(["Id"])
    .withColumn("current_illness_date", to_date("CURRENTILLNESSDATE"))
    .withColumn("service_date", to_date("SERVICEDATE"))
    .withColumn("last_billed_date_primary", to_date("LASTBILLEDDATE1"))
    .withColumn("last_billed_date_secondary", to_date("LASTBILLEDDATE2"))
    .withColumn("last_billed_date_patient", to_date("LASTBILLEDDATEP"))
    .select(
        col("Id").alias("claim_id"),
        col("PATIENTID").alias("patient_id"),
        col("PROVIDERID").alias("provider_id"),
        col("PRIMARYPATIENTINSURANCEID").alias("primary_insurance_id"),
        col("SECONDARYPATIENTINSURANCEID").alias("secondary_insurance_id"),
        col("DEPARTMENTID").alias("department_id"),
        col("PATIENTDEPARTMENTID").alias("patient_department_id"),

        "current_illness_date",
        "service_date",

        col("STATUS1").alias("primary_status"),
        col("STATUS2").alias("secondary_status"),
        col("STATUSP").alias("patient_status"),

        "last_billed_date_primary",
        "last_billed_date_secondary",
        "last_billed_date_patient",

        "source_file",
        "source_system",
        "load_timestamp"
    )
    .withColumn("silver_load_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit("bronze_to_silver"))
)

In [0]:
display(silver_claims)

In [0]:
(
    silver_claims.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{env_catalog}.silver.claims")
)

In [0]:
silver_claim_transactions = (
    claims_trans_df
    .dropDuplicates(["ID"])
    .withColumn("from_date", to_date("FROMDATE"))
    .withColumn("to_date", to_date("TODATE"))
    .select(
        col("ID").alias("transaction_id"),
        col("CLAIMID").alias("claim_id"),
        col("PATIENTID").alias("patient_id"),
        col("PROVIDERID").alias("provider_id"),

        col("TYPE").alias("transaction_type"),
        col("METHOD").alias("payment_method"),

        "from_date",
        "to_date",

        col("PLACEOFSERVICE").alias("place_of_service"),

        col("PROCEDURECODE").alias("procedure_code"),

        col("UNITS").alias("units"),

        col("AMOUNT").alias("amount"),

        col("UNITAMOUNT").alias("unit_amount"),

        col("PAYMENTS").alias("payments"),

        col("ADJUSTMENTS").alias("adjustments"),

        col("TRANSFERS").alias("transfers"),

        col("OUTSTANDING").alias("outstanding"),

        "source_file"
    )
    .withColumn("silver_load_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit("bronze_to_silver"))
)

In [0]:
display(silver_claim_transactions)

In [0]:
(
    silver_claim_transactions.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{env_catalog}.silver.claims_transactions"
    )
)